In [ ]:
import pandas as pd

prices = pd.read_csv("cleaned_stock_data.csv")

print(prices.head())

### MACD (MANUAL IMPLEMENTATION)

In [ ]:
# EMA calculation
def ema(series, span):
    return series.ewm(span=span, adjust=False).mean()

# MACD
prices['EMA_12'] = prices.groupby('ticker_encoded')['close'].transform(lambda x: ema(x, 12))
prices['EMA_26'] = prices.groupby('ticker_encoded')['close'].transform(lambda x: ema(x, 26))

prices['MACD'] = prices['EMA_12'] - prices['EMA_26']

# Signal line
prices['MACD_signal'] = prices.groupby('ticker_encoded')['MACD'].transform(lambda x: ema(x, 9))

### RSI (MANUAL)

In [ ]:
def compute_rsi(series, window=14):
    delta = series.diff()

    gain = (delta.where(delta > 0, 0)).rolling(window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window).mean()

    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))

    return rsi

prices['RSI'] = prices.groupby('ticker_encoded')['close'].transform(lambda x: compute_rsi(x))

### SIGNAL GENERATION

In [ ]:
def generate_signal(row):
    if (row['MACD'] > row['MACD_signal']) and (row['RSI'] < 30):
        return "Buy"
    elif (row['MACD'] < row['MACD_signal']) and (row['RSI'] > 70):
        return "Sell"
    else:
        return "Hold"

prices['signal'] = prices.apply(generate_signal, axis=1)

print(prices['signal'].value_counts())

### PREPARING DATA FOR ML

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
prices['signal_encoded'] = le.fit_transform(prices['signal'])

### Drop unnecessary columns

In [ ]:
prices = prices.drop(columns=['signal', 'date'], errors='ignore')
prices = prices.dropna()

### Define X and y

In [ ]:
X = prices.drop(columns=['signal_encoded'])
y = prices['signal_encoded']

### SPLIT DATA

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

### MODEL BUILDING
#### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=50)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

### SVM

In [ ]:
from sklearn.svm import SVC

svm = SVC()
svm.fit(X_train, y_train)

svm_pred = svm.predict(X_test)

### MODEL EVALUATION

In [ ]:
from sklearn.metrics import classification_report

print("Logistic Regression\n", classification_report(y_test, lr_pred))
print("Random Forest\n", classification_report(y_test, rf_pred))
print("SVM\n", classification_report(y_test, svm_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, rf_pred)

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix - Random Forest")
plt.show()